# Modeling exploration

Use this notebook to iterate on feature engineering and model behavior locally without Strava API calls.

## Setup

Run once from repo root:
```bash
pip install -r requirements-dev.txt
python -m ipykernel install --user --name train-tomorrow
```

To use live data, set `USE_LIVE_DATA = True` in the next cell. Requires a repo-root `.env` with `STRAVA_CLIENT_ID`, `STRAVA_CLIENT_SECRET`, `STRAVA_REFRESH_TOKEN`, and optionally `FORECAST_LAT`/`FORECAST_LON`.

In [ ]:
import logging
import os
import sys
import tempfile
from pathlib import Path
import pandas as pd
import xgboost as xgb


import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

sys.path.insert(0, str(Path.cwd().parent / "scripts"))  # allow importing sibling scripts modules

from blurb import generate_blurb, summarize_top_contributors, generate_gemini_prompt, generate_blurb_llm
from features import FEATURE_COLUMNS, prepare_datasets
from model import evaluate_models, feature_contributions, predict_tomorrow, train_and_save_models, walk_forward_evaluate
from model import DEFAULT_XGB_PARAMS
from strava_client import fetch_activities_dataframe, out_of_range_dates
from weather_client import DEFAULT_LAT, DEFAULT_LON, fetch_historical_weather, fetch_tomorrow_forecast

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
load_dotenv(Path.cwd().parent / ".env")  # repo root .env


True

In [2]:
USE_LIVE_DATA = False  # set True to pull real Strava activities + Open-Meteo forecast via .env secrets

if USE_LIVE_DATA:
    activities = fetch_activities_dataframe(days_back=365*2)
    weather = fetch_tomorrow_forecast()
    activity_dates = pd.to_datetime(activities['date'], errors='coerce').dt.date.dropna()
    historical_weather = (
        fetch_historical_weather(activity_dates.min(), activity_dates.max())
        if not activity_dates.empty
        else None
    )
    if historical_weather is not None:
        home_lat = float(os.getenv('FORECAST_LAT', DEFAULT_LAT))
        home_lon = float(os.getenv('FORECAST_LON', DEFAULT_LON))
        stale_dates = out_of_range_dates(activities, home_lat, home_lon)
        historical_weather = historical_weather.drop(index=list(stale_dates), errors='ignore')
    prepared = prepare_datasets(activities=activities, tomorrow_weather=weather, historical_weather=historical_weather)
    historical = prepared.historical
else:
    historical = pd.read_csv("local_data/historical.csv")



In [25]:
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold

X = historical[FEATURE_COLUMNS]
y = historical['will_train_tomorrow']

rfecv = RFECV(
    estimator=xgb.XGBClassifier(),
    step=1,
    cv=StratifiedKFold(5),
    scoring="accuracy",
    min_features_to_select=10,
)
rfecv.fit(X, y)

selected_features = list(X.columns[rfecv.support_])
print(f"Kept {len(selected_features)}/{len(FEATURE_COLUMNS)} features:", selected_features)

Kept 10/33 features: ['trained_days_7', 'trained_days_30', 'moving_time_chronic_28', 'today_relative_effort', 'dow_train_rate', 'day_of_week', 'forecast_temp_high', 'forecast_temp_low', 'forecast_precip_probability_vs_seasonal', 'forecast_precip_midday']


In [26]:
wf_all = walk_forward_evaluate(historical)
wf_all.mean()[["auc", "accuracy", "baseline_accuracy"]]

auc                  0.640607
accuracy             0.567123
baseline_accuracy    0.663014
dtype: float64

In [27]:
wf_new = walk_forward_evaluate(historical, features=selected_features)
wf_new.mean()[["auc", "accuracy", "baseline_accuracy"]]

auc                  0.682675
accuracy             0.646575
baseline_accuracy    0.663014
dtype: float64

In [34]:
model = xgb.XGBClassifier(**DEFAULT_XGB_PARAMS)
model.fit(X[selected_features],y)

importance_df = pd.DataFrame({
    'feature': selected_features,
    'classifier_importance': model.feature_importances_,
}).sort_values('classifier_importance', ascending=False).reset_index(drop=True)

importance_df

,feature,classifier_importance
0,trained_days_30,0.132114
1,trained_days_7,0.121986
2,forecast_precip_midday,0.117482
3,day_of_week,0.105642
4,dow_train_rate,0.095427
5,forecast_temp_high,0.090926
6,forecast_temp_low,0.090641
7,forecast_precip_probability_vs_seasonal,0.084452
8,today_relative_effort,0.081404
9,moving_time_chronic_28,0.079924


In [37]:
preds_in_sample = model.predict(X[selected_features])

In [41]:
preds_in_sample.sum()/len(preds_in_sample)

np.float64(0.5095890410958904)